# COMPAS Analysis — Follow-Along Notebook

This notebook reproduces the calculations in ProPublica's analysis of the COMPAS recidivism risk scores (["Machine Bias"](https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing/)), translated from R to Python. You may find it useful to keep [the methodology write-up](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm/) open in another tab.

**How this notebook works**

* All of the code is written for you. Run each cell in order (Shift + Enter) and **read the output carefully**.
* ProPublica's original notebook states its conclusions after the analysis. Here those conclusions have been removed. When you reach a **✍️ Your turn** cell, use the output above it to write the conclusion yourself. Fill in the sentence frame, and give the actual numbers.
* At the end of each major section there is a **💭 Reflection** prompt. Write a thoughtful paragraph (4–8 sentences) that connects what you found to the article you read.
* Write your answers in the **Your answer** cells (double-click a cell to edit it, Shift + Enter to render it).

In [ ]:
# If lifelines is not installed, uncomment the next line and run this cell once.
# %pip install lifelines

import math
import re
import statistics
import warnings
from csv import DictReader

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter, KaplanMeierFitter

from truth_tables import PeekyReader, Person, table, is_race, count, vtable, hightable, vhightable

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)

### Helper functions

These helpers stand in for R functions used in the original analysis. You don't need to understand every line, but read the docstrings so you know what each one does.

In [ ]:
def relative_risk(model, term):
    """Convert a logistic regression coefficient into an approximate relative risk.

    Logistic regression gives *odds ratios*. ProPublica converts them to "X times as likely"
    using the probability of a high score for the reference group (the intercept).
    """
    intercept = model.params['Intercept']
    control = math.exp(intercept) / (1 + math.exp(intercept))
    odds_ratio = math.exp(model.params[term])
    return odds_ratio / (1 - control + control * odds_ratio)


def concordance(data, risk):
    """Concordance (C-index) for counting-process survival data, matching R's coxph output.

    For every pair where one person recidivated while the other was still at risk, ask:
    did the model give the person who recidivated the higher risk? Ties count as half.
    0.5 = coin flip, 1.0 = perfect ranking.
    """
    s, e, d = (data[c].to_numpy() for c in ('start', 'end', 'event'))
    r = np.asarray(risk)
    agree = disagree = tied = 0
    for t in np.unique(e[d == 1]):
        died = (e == t) & (d == 1)
        at_risk = (s < t) & ((e > t) | ((e == t) & (d == 0)))
        rr = np.sort(r[at_risk])
        for ri in r[died]:
            lo, hi = np.searchsorted(rr, ri, 'left'), np.searchsorted(rr, ri, 'right')
            agree += lo
            disagree += len(rr) - hi
            tied += hi - lo
    return (agree + tied / 2) / (agree + disagree + tied)


def fit_cox(data, formula):
    """Fit a Cox proportional hazards model on start/end (counting-process) data and print a summary."""
    cph = CoxPHFitter()
    predictors = [c for c in data.columns if c in re.findall(r'\w+', formula)]
    cph.fit(data[['start', 'end', 'event'] + predictors],
            duration_col='end', event_col='event', entry_col='start', formula=formula)
    print(f"n = {len(data)}, number of events = {int(data.event.sum())}\n")
    print(cph.summary[['coef', 'exp(coef)', 'se(coef)', 'p']].round(5).to_string())
    print(f"\nConcordance = {concordance(data, cph.predict_partial_hazard(data)):.3f}")
    return cph


def km_by_score(data, ax=None, title=''):
    """Plot Kaplan-Meier survival curves (share who have NOT recidivated) for each score level."""
    ax = ax or plt.gca()
    for level in ['Low', 'Medium', 'High']:
        sub = data[data.score_factor == level]
        KaplanMeierFitter().fit(sub.end, sub.event, entry=sub.start, label=level).plot_survival_function(ax=ax)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.set_xlabel('Days')
    ax.set_ylabel('Share not yet recidivated')
    return ax


def survival_at(data, days=730):
    """Kaplan-Meier estimate of the share of each score group that has NOT recidivated after `days`."""
    rows = {}
    for level in ['Low', 'Medium', 'High']:
        sub = data[data.score_factor == level]
        kmf = KaplanMeierFitter().fit(sub.end, sub.event, entry=sub.start)
        rows[level] = round(float(kmf.survival_function_at_times(days).iloc[0]), 3)
    return pd.Series(rows, name=f'survival at {days} days')


def error_rates(recid, surv, prefix=''):
    """Return (false positive rate, false negative rate) in percent, using the same rules as truth_tables.table."""
    fp = count(lambda i: getattr(i, prefix + 'high'), surv)
    fn = count(lambda i: getattr(i, prefix + 'low'), recid)
    return fp / len(surv) * 100, fn / len(recid) * 100

## Loading the Data

We select fields for severity of charge, number of priors, demographics, age, sex, COMPAS scores, and whether each person was accused of a crime within two years.

In [ ]:
raw_data = pd.read_csv('./compas-scores-two-years.csv')
len(raw_data)

However, not all of the rows are usable for the first round of analysis. ProPublica removed rows for several reasons:

* If the charge date of a defendant's COMPAS-scored crime was not within 30 days of when the person was arrested, we assume that, because of data quality issues, we do not have the right offense.
* The recidivist flag `is_recid` was coded as -1 if no COMPAS case could be found at all.
* Ordinary traffic offenses (`c_charge_degree` of `'O'`) will not result in jail time, so they are removed (there are only two).
* The underlying Broward County data was filtered to include only people who either recidivated within two years or had at least two years outside of a correctional facility.

In [ ]:
columns = ['age', 'c_charge_degree', 'race', 'age_cat', 'score_text', 'sex', 'priors_count',
           'days_b_screening_arrest', 'decile_score', 'is_recid', 'two_year_recid', 'c_jail_in', 'c_jail_out']

df = raw_data[columns]
df = df[(df.days_b_screening_arrest <= 30) &
        (df.days_b_screening_arrest >= -30) &
        (df.is_recid != -1) &
        (df.c_charge_degree != 'O') &
        (df.score_text != 'N/A')].copy()
len(df)

How does a defendant's COMPAS decile score relate to how long they stayed in jail?

In [ ]:
df['length_of_stay'] = (pd.to_datetime(df.c_jail_out).dt.normalize() -
                        pd.to_datetime(df.c_jail_in).dt.normalize()).dt.days
df.length_of_stay.corr(df.decile_score)

### ✍️ Your turn — Question 1

Interpret the correlation coefficient above.

> Higher COMPAS scores are **(strongly / slightly / not)** correlated with a **(longer / shorter)** length of stay (r = ____).

*Hint:* a correlation of 1 is a perfect positive relationship and 0 is no relationship. Where does this value fall?

**Your answer (Q1):**

*Double-click to write your answer here.*

After filtering, we have the following demographic breakdown:

In [ ]:
df.age_cat.value_counts()

In [ ]:
race_counts = df.race.value_counts()
pd.DataFrame({'count': race_counts, 'percent': (race_counts / len(df) * 100).round(2)})

In [ ]:
df.score_text.value_counts()

In [ ]:
pd.crosstab(df.sex, df.race)

In [ ]:
sex_counts = df.sex.value_counts()
pd.DataFrame({'count': sex_counts, 'percent': (sex_counts / len(df) * 100).round(2)})

In [ ]:
n_recid = (df.two_year_recid == 1).sum()
print(f"Recidivated within two years: {n_recid} ({n_recid / len(df) * 100:.2f}%)")

Judges are often shown two kinds of COMPAS output: a category (Low, Medium or High risk) and a corresponding **decile score** from 1 (lowest risk) to 10 (highest risk). Let's compare the distribution of decile scores for Black and white defendants.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (race, label) in zip(axes, [('African-American', 'Black'), ('Caucasian', 'White')]):
    counts = df[df.race == race].decile_score.value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='0.35')
    ax.set_ylim(0, 650)
    ax.set_xlabel('Decile Score')
    ax.set_title(f"{label} Defendants' Decile Scores")
axes[0].set_ylabel('Count')
plt.show()

In [ ]:
pd.crosstab(df.decile_score, df.race)

### ✍️ Your turn — Question 2

Compare the two histograms.

> For **white** defendants, the number of people at each decile score **(rises / falls / stays about the same)** as the score increases. For **Black** defendants, the scores are **(concentrated at the low end / spread fairly evenly / concentrated at the high end)**.

Then, in one sentence: why can't we conclude from these histograms alone that COMPAS is biased? (What else could explain the difference?)

**Your answer (Q2):**

*Double-click to write your answer here.*

## 💭 Reflection 1: The data

Before running any models, ProPublica made several choices about which defendants to **keep** and which to **drop** (the 30-day window, missing cases, traffic offenses, two-year follow-up).

* Pick one of these filters. Who gets removed by it, and why might that matter for the conclusions?
* The article focuses on Broward County, Florida. What are the limits of generalizing from one county?
* The outcome we call "recidivism" is really "was *charged* with a new crime within two years." How might that differ from "committed a new crime," and could the difference vary by race?

**Your reflection (R1):**

*Double-click to write your reflection here.*

## Racial Bias in COMPAS

Our first question is whether there is a significant difference in COMPAS scores between races. To answer it we run a **logistic regression**. The outcome is whether someone got a Medium/High score (vs. Low), and the predictors are race, sex, age, number of priors, charge degree (felony vs. misdemeanor), and whether the person actually reoffended within two years.

The **reference groups** (what everyone is compared against) are: white (`Caucasian`), male, age 25–45, and felony charge.

In [ ]:
df['high_score'] = (df.score_text != 'Low').astype(int)

formula = ("high_score ~ C(sex, Treatment('Male')) + C(age_cat, Treatment('25 - 45'))"
           " + C(race, Treatment('Caucasian')) + priors_count"
           " + C(c_charge_degree, Treatment('F')) + two_year_recid")
model = smf.logit(formula, data=df).fit(disp=0)
print(model.summary())

Logistic regression coefficients are in "log-odds," which are hard to interpret directly. The `relative_risk` helper converts a coefficient into "how many times as likely to get a higher score, compared to the reference group."

In [ ]:
black_rr = relative_risk(model, "C(race, Treatment('Caucasian'))[T.African-American]")
print(f"Black vs. white: {black_rr:.3f}")

### ✍️ Your turn — Question 3

Write the main finding of this regression.

> Black defendants are ____% **(more / less)** likely than white defendants to receive a higher score, correcting for the seriousness of their crime, ________________, and ________________.

*Hint:* a relative risk of 1.25 means "25% more likely." Look at the list of other predictors in the model to fill in what was "corrected for."

**Your answer (Q3):**

*Double-click to write your answer here.*

In [ ]:
female_rr = relative_risk(model, "C(sex, Treatment('Male'))[T.Female]")
young_rr = relative_risk(model, "C(age_cat, Treatment('25 - 45'))[T.Less than 25]")
print(f"Women vs. men:            {female_rr:.3f}")
print(f"Under 25 vs. age 25-45:   {young_rr:.3f}")

### ✍️ Your turn — Question 4

Now do the same for sex and age.

> Women are ____% **(more / less)** likely than men to get a higher score.
>
> People under 25 are ____ times as likely to get a higher score as middle-aged (25–45) defendants.

Which of these two results surprised you more, and why?

**Your answer (Q4):**

*Double-click to write your answer here.*

### Risk of Violent Recidivism

COMPAS also produces a score that aims to measure a person's risk of **violent** recidivism. Its overall accuracy is similar to the general recidivism score. As before, we use logistic regression to test for racial bias.

In [ ]:
raw_data = pd.read_csv('./compas-scores-two-years-violent.csv')
len(raw_data)

In [ ]:
columns = ['age', 'c_charge_degree', 'race', 'age_cat', 'v_score_text', 'sex', 'priors_count',
           'days_b_screening_arrest', 'v_decile_score', 'is_recid', 'two_year_recid']

vdf = raw_data[columns]
vdf = vdf[(vdf.days_b_screening_arrest <= 30) &
          (vdf.days_b_screening_arrest >= -30) &
          (vdf.is_recid != -1) &
          (vdf.c_charge_degree != 'O') &
          (vdf.v_score_text != 'N/A')].copy()
len(vdf)

In [ ]:
vdf.age_cat.value_counts()

In [ ]:
vdf.race.value_counts()

In [ ]:
vdf.v_score_text.value_counts()

In [ ]:
n_recid = (vdf.two_year_recid == 1).sum()
print(f"Recidivated (violent) within two years: {n_recid} ({n_recid / len(vdf) * 100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (race, label) in zip(axes, [('African-American', 'Black'), ('Caucasian', 'White')]):
    counts = vdf[vdf.race == race].v_decile_score.value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='0.35')
    ax.set_ylim(0, 700)
    ax.set_xlabel('Violent Decile Score')
    ax.set_title(f"{label} Defendants' Violent Decile Scores")
axes[0].set_ylabel('Count')
plt.show()

In [ ]:
vdf['high_score'] = (vdf.v_score_text != 'Low').astype(int)
vmodel = smf.logit(formula, data=vdf).fit(disp=0)
print(vmodel.summary())

In [ ]:
v_black_rr = relative_risk(vmodel, "C(race, Treatment('Caucasian'))[T.African-American]")
v_young_rr = relative_risk(vmodel, "C(age_cat, Treatment('25 - 45'))[T.Less than 25]")
print(f"Black vs. white:          {v_black_rr:.3f}")
print(f"Under 25 vs. age 25-45:   {v_young_rr:.3f}")

### ✍️ Your turn — Question 5

Summarize the violent-score regression.

> For the violent score, Black defendants are ____% more likely than white defendants to get a higher score, controlling for the same factors.
>
> Defendants under 25 are ____ times as likely to get a higher score as middle-aged defendants.

Is the racial gap larger or smaller than for the general recidivism score?

**Your answer (Q5):**

*Double-click to write your answer here.*

## 💭 Reflection 2: Racial bias in scores

The regressions "control for" prior record, charge degree, age, sex, and actual two-year recidivism.

* In your own words, what does it mean that the race effect remains *after* controlling for these things?
* COMPAS does **not** use race as an input. How could the scores still differ by race? (Think about the questionnaire items described in the article.)
* What can this regression **not** tell us? Name at least one limitation.

**Your reflection (R2):**

*Double-click to write your reflection here.*

## Predictive Accuracy of COMPAS

To test whether COMPAS does an accurate job of sorting people into Low, Medium and High risk, ProPublica ran a **Cox proportional hazards model**. Northpointe, the company that created COMPAS, also ran a Cox model in its [validation study](http://cjb.sagepub.com/content/36/1/21.abstract).

A Cox model compares how quickly different groups "fail" (here, reoffend). A **hazard ratio** of 2 means a group reoffends at twice the rate of the reference group (here, Low scorers) at any given time. The **concordance** measures how well the scores *rank* people: 0.5 is a coin flip and 1.0 is perfect.

The data has one row per period a person spent outside jail, and people are removed from the "at risk" pool while they are incarcerated. Due to errors in the underlying jail data, we filter out a small number of rows whose end date is not after the start date.

In [ ]:
def load_cox_data(path):
    data = pd.read_csv(path)
    data = data[(data.score_text != 'N/A') & data.score_text.notna() & (data.end > data.start)].copy()
    races = ['Caucasian', 'African-American', 'Asian', 'Hispanic', 'Native American', 'Other']
    data['race_factor'] = pd.Categorical(data.race, categories=races)       # reference = Caucasian
    data['score_factor'] = pd.Categorical(data.score_text, categories=['Low', 'Medium', 'High'])  # reference = Low
    return data

data = load_cox_data('./cox-parsed.csv')
grp = data.drop_duplicates('id')
len(grp)

In [ ]:
grp.score_factor.value_counts()

In [ ]:
grp.race_factor.value_counts()

In [ ]:
cox_score = fit_cox(data, 'score_factor')

In [ ]:
cox_decile = fit_cox(data, 'decile_score')

### ✍️ Your turn — Question 6

Interpret the two models above.

> People placed in the **High** category are ____ times as likely to recidivate as people in the Low category.
>
> The concordance of the Low/Medium/High categories is ____%, which is **(higher / lower)** than the 68% reported in Northpointe's validation study. The 1–10 decile score has a concordance of ____%.

*Hint:* the hazard ratio is in the `exp(coef)` column.

**Your answer (Q6):**

*Double-click to write your answer here.*

We can test whether the algorithm behaves differently across races by including a **race × score interaction** in the Cox model.

In [ ]:
cox_interaction = fit_cox(data, 'race_factor * score_factor')

In [ ]:
params = cox_interaction.params_
high = params['score_factor[T.High]']
medium = params['score_factor[T.Medium]']
black_high = params['race_factor[T.African-American]:score_factor[T.High]']
black_medium = params['race_factor[T.African-American]:score_factor[T.Medium]']

print(f"Black High Hazard:   {math.exp(high + black_high):.2f}")
print(f"White High Hazard:   {math.exp(high):.2f}")
print(f"Black Medium Hazard: {math.exp(medium + black_medium):.2f}")
print(f"White Medium Hazard: {math.exp(medium):.2f}")

### ✍️ Your turn — Question 7

Each hazard above compares High (or Medium) scorers to **Low scorers of the same race**.

> High-risk white defendants are ____ times as likely to recidivate as low-risk white defendants, while high-risk Black defendants are ____ times as likely as low-risk Black defendants.

What does it mean that the "High" label separates people *less* sharply for Black defendants? How does this connect to the logistic regression results?

**Your answer (Q7):**

*Double-click to write your answer here.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
km_by_score(data, ax, 'Overall')
plt.show()

In [ ]:
white = data[data.race == 'Caucasian']
black = data[data.race == 'African-American']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
km_by_score(white, axes[0], 'White defendants')
km_by_score(black, axes[1], 'Black defendants')
plt.show()

In [ ]:
pd.DataFrame({'Overall': survival_at(data),
              'Black': survival_at(black),
              'White': survival_at(white)})

### ✍️ Your turn — Question 8

These **Kaplan-Meier** curves show the share of people who have *not* yet reoffended over time. The table gives that share at two years (730 days).

> According to these race-specific plots, Black defendants **(do / do not)** recidivate at higher rates than white defendants. For example, ____% of Black Low-risk defendants had not reoffended after two years, compared to ____% of white Low-risk defendants.

*Hint:* recidivism rate = 1 − survival.

**Your answer (Q8):**

*Double-click to write your answer here.*

In [ ]:
print('White defendants')
_ = fit_cox(white, 'score_factor')
print('\n\nBlack defendants')
_ = fit_cox(black, 'score_factor')

### ✍️ Your turn — Question 9

> When fit separately for each race, the concordance is ____ for white defendants and ____ for Black defendants. These values are **(very similar / very different)**.

Does this mean COMPAS is equally fair to both groups? Why or why not?

**Your answer (Q9):**

*Double-click to write your answer here.*

### Violent recidivism

In [ ]:
violent_data = load_cox_data('./cox-violent-parsed.csv')
vgrp = violent_data.drop_duplicates('id')
print(len(vgrp))
cox_violent = fit_cox(violent_data, 'score_factor')

In [ ]:
cox_violent_interaction = fit_cox(violent_data, 'race_factor + race_factor * score_factor')

In [ ]:
print('Black defendants')
_ = fit_cox(violent_data[violent_data.race == 'African-American'], 'score_factor')
print('\n\nWhite defendants')
_ = fit_cox(violent_data[violent_data.race == 'Caucasian'], 'score_factor')

In [ ]:
vwhite = violent_data[violent_data.race == 'Caucasian']
vblack = violent_data[violent_data.race == 'African-American']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
km_by_score(vwhite, axes[0], 'White defendants (violent)')
km_by_score(vblack, axes[1], 'Black defendants (violent)')
plt.show()

### ✍️ Your turn — Question 10

> COMPAS's violent recidivism score has an overall concordance of ____%, which is **(slightly higher / slightly lower)** than the general score.
>
> In the interaction model, the coefficient for Black defendants with High violent scores (`race_factor[T.African-American]:score_factor[T.High]`) **(is / is not)** statistically significant (p = ____).

What does that p-value mean for whether the violent score behaves differently by race in this model?

**Your answer (Q10):**

*Double-click to write your answer here.*

## 💭 Reflection 3: Predictive accuracy

COMPAS correctly ranks pairs of defendants about 64% of the time (a coin flip would get 50%).

* Is that "good enough" to inform decisions about bail, sentencing, or parole? Explain your reasoning.
* Who should decide what level of accuracy is acceptable: the company, judges, legislators, or the public?
* The article contrasts ProPublica's 63.6% with Northpointe's 68%. Why might two analyses of the same tool get different numbers?

**Your reflection (R3):**

*Double-click to write your reflection here.*

## Directions of the Racial Bias

The analysis above shows that COMPAS treats Black and white defendants differently, but we haven't yet looked at the **direction** of its mistakes. A risk tool can make two kinds of errors:

|                     | Labeled Low | Labeled Medium/High |
|---------------------|-------------|---------------------|
| **Did not reoffend** | ✅ True negative | ❌ **False positive** |
| **Reoffended**       | ❌ **False negative** | ✅ True positive |

* **False positive rate** = of the people who did *not* reoffend, the share labeled higher risk.
* **False negative rate** = of the people who *did* reoffend, the share labeled low risk.

This part of the analysis uses ProPublica's original Python helper file, `truth_tables.py`.

In [ ]:
people = []
with open('./cox-parsed.csv') as f:
    reader = PeekyReader(DictReader(f))
    try:
        while True:
            p = Person(reader)
            if p.valid:
                people.append(p)
    except StopIteration:
        pass

pop = list(filter(lambda i: ((i.recidivist == True and i.lifetime <= 730) or
                              i.lifetime > 730), list(filter(lambda x: x.score_valid, people))))
recid = list(filter(lambda i: i.recidivist == True and i.lifetime <= 730, pop))
rset = set(recid)
surv = [i for i in pop if i not in rset]

In [ ]:
print('All defendants')
table(list(recid), list(surv))

In [ ]:
print(f"Total pop: {len(pop)}")
lifetimes = [i.lifetime for i in pop]
print(f"Average followup time {statistics.mean(lifetimes):.2f} (sd {statistics.stdev(lifetimes):.2f})")
print(f"Median followup time {statistics.median(lifetimes)}")

In [ ]:
print('Black defendants')
is_afam = is_race('African-American')
table(list(filter(is_afam, recid)), list(filter(is_afam, surv)))

In [ ]:
print('White defendants')
is_white = is_race('Caucasian')
table(list(filter(is_white, recid)), list(filter(is_white, surv)))

In [ ]:
black_fpr, black_fnr = error_rates(list(filter(is_afam, recid)), list(filter(is_afam, surv)))
white_fpr, white_fnr = error_rates(list(filter(is_white, recid)), list(filter(is_white, surv)))

print(f"False positive rate  Black: {black_fpr:.2f}%   White: {white_fpr:.2f}%   ratio: {black_fpr / white_fpr:.2f}")
print(f"False negative rate  Black: {black_fnr:.2f}%   White: {white_fnr:.2f}%   ratio: {white_fnr / black_fnr:.2f}")

### ✍️ Your turn — Question 11

Use the three truth tables and the summary cell above.

> Overall, the false positive rate is ____%. That number is **(higher / lower)** for Black defendants at ____% and **(higher / lower)** for white defendants at ____%.
>
> This means that under COMPAS, Black defendants are ____% more likely than white defendants to get a higher score and **not** go on to commit more crimes within two years.
>
> In the other direction, COMPAS misclassifies white reoffenders as low risk ____% more often than Black reoffenders.

*Hint:* a ratio of 1.5 means "50% more likely."

**Your answer (Q11):**

*Double-click to write your answer here.*

What if only the **High** category counts as "high risk" (and Medium is grouped with Low)?

In [ ]:
print('White defendants (High only)')
hightable(list(filter(is_white, recid)), list(filter(is_white, surv)))

In [ ]:
print('Black defendants (High only)')
hightable(list(filter(is_afam, recid)), list(filter(is_afam, surv)))

### ✍️ Your turn — Question 12

> When only the **High** category counts as high risk, the false positive rate is ____% for Black defendants and ____% for white defendants.

Does the gap between the groups shrink, grow, or stay about the same in *relative* terms (compute the ratio)? Why might the choice of cut-off matter for a judge?

**Your answer (Q12):**

*Double-click to write your answer here.*

### Violent recidivism

COMPAS's violent recidivism score has a similar overall accuracy to the general score. Let's build the same truth tables.

In [ ]:
vpeople = []
with open('./cox-violent-parsed.csv') as f:
    reader = PeekyReader(DictReader(f))
    try:
        while True:
            p = Person(reader)
            if p.valid:
                vpeople.append(p)
    except StopIteration:
        pass

vpop = list(filter(lambda i: ((i.violent_recidivist == True and i.lifetime <= 730) or
                              i.lifetime > 730), list(filter(lambda x: x.vscore_valid, vpeople))))
vrecid = list(filter(lambda i: i.violent_recidivist == True and i.lifetime <= 730, vpeople))
vrset = set(vrecid)
vsurv = [i for i in vpop if i not in vrset]

In [ ]:
print('All defendants')
vtable(list(vrecid), list(vsurv))

In [ ]:
print('Black defendants')
vtable(list(filter(is_afam, vrecid)), list(filter(is_afam, vsurv)))

In [ ]:
print('White defendants')
vtable(list(filter(is_white, vrecid)), list(filter(is_white, vsurv)))

In [ ]:
vblack_fpr, vblack_fnr = error_rates(list(filter(is_afam, vrecid)), list(filter(is_afam, vsurv)), prefix='v')
vwhite_fpr, vwhite_fnr = error_rates(list(filter(is_white, vrecid)), list(filter(is_white, vsurv)), prefix='v')

print(f"False positive rate  Black: {vblack_fpr:.2f}%   White: {vwhite_fpr:.2f}%   ratio: {vblack_fpr / vwhite_fpr:.2f}")
print(f"False negative rate  Black: {vblack_fnr:.2f}%   White: {vwhite_fnr:.2f}%   ratio: {vwhite_fnr / vblack_fnr:.2f}")

### ✍️ Your turn — Question 13

> For the violent score, Black defendants are about ____ times as likely as white defendants to be false positives (labeled higher risk but did not reoffend violently).
>
> White defendants are ____% more likely than Black defendants to get a lower score and go on to commit another violent crime.

Is the pattern the same as for the general recidivism score?

**Your answer (Q13):**

*Double-click to write your answer here.*

## 💭 Reflection 4: Fairness: whose definition?

Northpointe responded to ProPublica by arguing that COMPAS is fair because it is **calibrated**: a given score means roughly the same chance of reoffending regardless of race (you saw this in the Kaplan-Meier curves). ProPublica argues it is unfair because the **error rates** (false positives and false negatives) differ sharply by race.

* Explain each side's definition of fairness in your own words, using numbers from this notebook.
* Black and white defendants in this data reoffend at different overall rates. Researchers have shown that when base rates differ, a tool **cannot** be both calibrated and have equal error rates (unless it is perfect). Which kind of fairness do you think matters more in a courtroom, and why?
* Who bears the cost of a false positive? Who bears the cost of a false negative?

**Your reflection (R4):**

*Double-click to write your reflection here.*

## Gender Differences in COMPAS Scores

In terms of underlying recidivism rates, we can look at gender-specific Kaplan-Meier estimates.

In [ ]:
female = data[data.sex == 'Female']
male = data[data.sex == 'Male']

pd.DataFrame({'Female': survival_at(female), 'Male': survival_at(male)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
km_by_score(female, axes[0], 'Female')
km_by_score(male, axes[1], 'Male')
plt.show()

### ✍️ Your turn — Question 14

Compare the curves and the two-year survival table.

> There is a **(small / striking)** difference between women and men. A woman labeled **High** risk has a two-year survival of ____%, which is closest to a man labeled ____ risk (____%).
>
> In other words, the COMPAS score treats a High-risk woman the same as a ____-risk man.

**Your answer (Q14):**

*Double-click to write your answer here.*

## 💭 Reflection 5: Gender

COMPAS gives women and men with the same score different real-world risks.

* If including sex as an input would make the tool more accurate for women, should it be included? Why or why not?
* Would your answer be the same for race? What is different (or not) about the two cases, legally and ethically?

**Your reflection (R5):**

*Double-click to write your reflection here.*

## 💭 Reflection 6: Closing reflection

You have now reproduced the core of ProPublica's analysis yourself.

* Did running the numbers change how you think about the article? What, if anything, surprised you?
* If you were advising Broward County on whether to keep using COMPAS, what would you recommend, and what evidence from this notebook would you point to?

**Your reflection (R6):**

*Double-click to write your reflection here.*